# Streaming & Batch Inference in LangChain
### Get tokens as they arrive and process many inputs at once

---
**Topics Covered:**
- `stream()` — iterate over token chunks as they're generated
- Streaming an entire LCEL chain (prompt → llm → parser)
- `astream()` — async streaming with `asyncio`
- `batch()` — run many prompts in one call
- `abatch()` — async batch processing
- Streaming with events via `astream_events()`


In [ ]:
import os, asyncio, time
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq", temperature=0.7)

## 1. Basic Streaming with `stream()`

`model.stream()` returns an iterator that yields `AIMessageChunk` objects.  
Print each chunk's `.content` as it arrives for a real-time typing effect.

In [ ]:
print("Streaming response:\n")

full_text = ""
for chunk in llm.stream("Write a short origin story for a robot that learns to paint."):
    print(chunk.content, end="", flush=True)
    full_text += chunk.content

print(f"\n\n--- Total characters: {len(full_text)} ---")

## 2. Streaming Through a Full LCEL Chain

When you call `.stream()` on a chain, chunks flow through every step and you receive the final output progressively.

In [ ]:
story_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative sci-fi writer. Write imaginative, vivid stories."),
    ("human", "Write a 120-word sci-fi microstory set on {planet}. Theme: {theme}.")
])

story_chain = story_prompt | llm | StrOutputParser()

print("Streaming chain output:\n")
for token in story_chain.stream({"planet": "Europa", "theme": "first contact"}):
    print(token, end="", flush=True)

print()

## 3. Collect All Chunks Into a List

Use `list(chain.stream(...))` when you want all the chunks after generation finishes.

In [ ]:
chunks = list(
    story_chain.stream({"planet": "Titan", "theme": "lost civilisation"})
)

print(f"Number of chunks received: {len(chunks)}")
print(f"First chunk: '{chunks[0]}'")
print(f"Last chunk:  '{chunks[-1]}'")

# Reconstruct the full story
full_story = "".join(chunks)
print("\nFull story:\n", full_story)

## 4. Async Streaming with `astream()`

Use `astream()` inside an `async` function to stream without blocking the event loop.

In [ ]:
async def stream_poem(topic: str):
    print(f"Streaming poem about '{topic}':\n")
    async for token in story_chain.astream({"planet": topic, "theme": "solitude"}):
        print(token, end="", flush=True)
    print()

# Jupyter / IPython can run async directly
await stream_poem("Mars")

## 5. Batch Processing with `batch()`

`batch()` accepts a list of inputs and returns a list of outputs.  
LangChain runs them concurrently (up to `max_concurrency`) under the hood.

In [ ]:
fact_prompt = ChatPromptTemplate.from_messages([
    ("system", "Reply with exactly ONE surprising fact. One sentence only."),
    ("human", "Give me a surprising fact about {subject}.")
])
fact_chain = fact_prompt | llm | StrOutputParser()

subjects = [
    {"subject": "octopuses"},
    {"subject": "the number zero"},
    {"subject": "the speed of light"},
    {"subject": "honey"},
    {"subject": "the human brain"},
]

start = time.time()
facts = fact_chain.batch(subjects, config={"max_concurrency": 5})
elapsed = time.time() - start

print(f"Processed {len(facts)} queries in {elapsed:.2f}s\n")
for inp, fact in zip(subjects, facts):
    print(f"[{inp['subject']:20s}] {fact}")

## 6. Async Batch with `abatch()`

In [ ]:
async def run_batch_async():
    topics = [
        {"subject": "quantum entanglement"},
        {"subject": "black holes"},
        {"subject": "dark matter"},
    ]
    results = await fact_chain.abatch(topics)
    for t, r in zip(topics, results):
        print(f"{t['subject']:25s}: {r}")

await run_batch_async()

## 7. `astream_events()` — Fine-Grained Event Stream

`astream_events()` gives you lifecycle events: `on_chain_start`, `on_llm_stream`, `on_chain_end`, etc.  
Useful for building progress indicators or logging.

In [ ]:
async def watch_events():
    event_counts = {}
    async for event in story_chain.astream_events(
        {"planet": "Venus", "theme": "terraforming"},
        version="v2"
    ):
        name = event["event"]
        event_counts[name] = event_counts.get(name, 0) + 1

        # Print LLM token chunks only
        if name == "on_chat_model_stream":
            token = event["data"]["chunk"].content
            print(token, end="", flush=True)

    print(f"\n\nEvent summary: {event_counts}")

await watch_events()

## 8. Timing: stream vs invoke vs batch

In [ ]:
import time

query = {"subject": "the Milky Way galaxy"}

# invoke — wait for full response
t0 = time.time()
_ = fact_chain.invoke(query)
invoke_time = time.time() - t0

# stream — time to first token
t0 = time.time()
first_token_time = None
for tok in fact_chain.stream(query):
    if first_token_time is None and tok:
        first_token_time = time.time() - t0
stream_total = time.time() - t0

print(f"invoke total  : {invoke_time:.2f}s")
print(f"stream TTFT   : {first_token_time:.2f}s  (time to first token)")
print(f"stream total  : {stream_total:.2f}s")

---
### Summary

| Method | When to use |
|--------|-------------|
| `invoke(input)` | Single input, wait for full response |
| `stream(input)` | Single input, get tokens progressively |
| `batch(inputs)` | Many inputs, concurrent processing |
| `ainvoke / astream / abatch` | Async versions of the above |
| `astream_events()` | Detailed lifecycle events per step |

**Series complete!** You've covered: fundamentals → prompts → tools → LCEL → streaming.